In [1]:
import dask_geopandas as dgpd
import geopandas as gpd
import rasterio
from rasterio.sample import sample_gen
from pathlib import Path
import numpy as np
import pandas as pd
import dask

In [2]:
terrain_attributes = dgpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_terrain_attributes.parquet")

In [3]:
geomorphon_gdf = dgpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_geomorphons.parquet")

In [4]:
hand_gdf = dgpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_HAND.parquet")


In [5]:
print(terrain_attributes.head(1))
print(geomorphon_gdf.head(1))
print(hand_gdf.head(1))
print(terrain_attributes.columns)
print(geomorphon_gdf.columns)
print(hand_gdf.columns)


                               region  sc_orient  track  segment_dist  \
2018-11-04 01:05:31.626116096     6.0        1.0    1.0  1.472979e+07   

                               solar_elevation  segment_id  background_rate  \
2018-11-04 01:05:31.626116096       -40.574768    735401.0      2756.054548   

                               cycle  pair    rgt  ...  srtm_dem_roughness  \
2018-11-04 01:05:31.626116096    1.0   0.0  556.0  ...                24.0   

                               srtm_dem_slope  srtm_dem_tpi  srtm_dem_tri  \
2018-11-04 01:05:31.626116096       18.294834         1.375      22.15852   

                               tan_dem_aspect  tan_dem_curvature  \
2018-11-04 01:05:31.626116096       137.23674           1.002909   

                               tan_dem_roughness  tan_dem_slope  tan_dem_tpi  \
2018-11-04 01:05:31.626116096          23.900085      17.826292     2.318726   

                               tan_dem_tri  
2018-11-04 01:05:31.626116096    22.068

In [6]:
dask.config.set(scheduler="threads")

In [ ]:
import dask
import dask_geopandas as dgpd
from dask.distributed import Client
client = Client()
dask.config.set(scheduler="threads")

terrain = dgpd.read_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_terrain_attributes.parquet").set_index('time')

geomorph = dgpd.read_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_geomorphons.parquet").set_index('time')
hand = dgpd.read_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_HAND.parquet").set_index('time')


# 2. Нові колонки
geom_cols = [c for c in geomorph.columns if c not in terrain.columns and c != 'geometry']
hand_cols = [c for c in hand.columns if c not in terrain.columns and c != 'geometry']
terrain = terrain.repartition(npartitions=50)
# 3. Join-им лінено
result = terrain

if geom_cols:
    result = result.join(geomorph[geom_cols], how='left')
if hand_cols:
    result = result.join(hand[hand_cols], how='left')



from dask.diagnostics import ProgressBar

with ProgressBar():
    result.to_parquet(
        "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/merged_icesat2_final.parquet",
        compute=True
    )

[########################################] | 100% Completed | 4.27 ss
[                                        ] | 0% Completed | 3.26 s ms

In [ ]:
print(result.index)

In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Зчитуємо всі дані в пам'ять
terrain = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_terrain_attributes.parquet")
geomorph = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_geomorphons.parquet")
hand = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_HAND.parquet")

# 2. Визначаємо нові колонки (уникнути дублювання)
geom_cols = [c for c in geomorph.columns if c not in terrain.columns and c != 'geometry']
hand_cols = [c for c in hand.columns if c not in terrain.columns and c != 'geometry']

# 3. Merge/join — по колонці 'time', щоб не дублювати geometry!
# Головний датафрейм — terrain, додаємо лише унікальні колонки
result = terrain.set_index('time')
if geom_cols:
    result = result.join(geomorph.set_index('time')[geom_cols], how='left')
if hand_cols:
    result = result.join(hand.set_index('time')[hand_cols], how='left')

# (Опційно) повертаємо time як колонку
result = result.reset_index()

# 4. Зберігаємо у Parquet
result.to_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/merged_icesat2_final.parquet")
